In [ ]:
import os
from langchain_ollama import ChatOllama
from langchain_core.documents import Document
from langchain_experimental.graph_transformers import LLMGraphTransformer

# LLM_MODEL_NAME="jaro-jm/llm:latest"
LLM_MODEL_NAME="SpeakLeash/bielik-11b-v2.3-instruct:Q8_0"
os.environ["HSA_OVERRIDE_GFX_VERSION"]="11.0.0"

llm_text = ChatOllama(model=LLM_MODEL_NAME, temperature=0, reasoning=False, num_ctx=25000, num_predict=25000)

text = """Co Trump uważa o wojnie w Ukrainie?"""

In [ ]:
llm_transformer = LLMGraphTransformer(
    llm=llm_text,
    node_properties=True,
    relationship_properties=True,
    strict_mode=False,
    additional_instructions="""Rules:
- Return everything in POLISH: head, head_type, relation, tail and tail_type must be written in Polish.
- Extract 2-5 most important nodes (entities, concepts, events) and 2-3 most important relationships between them from provided user's question.
- Do not create meta-nodes (e.g., Autor, Tekst, Artykuł). First-person statements should be reduced to the underlying objective fact connecting real entities.""",
)

graph_documents = llm_transformer.convert_to_graph_documents([Document(text)])

In [ ]:
for doc_idx, graph_doc in enumerate(graph_documents, start=1):
    print(f"\n=== DOCUMENT {doc_idx} ===")
    print("NODES:")
    for i, node in enumerate(graph_doc.nodes, start=1):
        print(f"  {i:02d}. {node.id} ({node.type})")

    print("RELATIONSHIPS:")
    for i, rel in enumerate(graph_doc.relationships, start=1):
        source = rel.source.id
        target = rel.target.id
        rel_type = rel.type
        print(f"  {i:02d}. {source} ──[{rel_type}]──▶ {target}")